# 4.1 预训练目标与数据 / Pretraining objective and data

参数怎样从随机数变成有用的权重。看训练数据怎样构造、损失怎样计算、优化器每一步做了什么。

How parameters turn from random numbers into useful weights. How training data is built, how the loss is computed and what the optimizer does at each step.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 构造错位一格的 batch 与损失 / Build shifted batches and the loss

实现预训练数据的取样、带忽略位置的逐 token 损失，以及两个小的估算函数。测试会验证目标正好是输入错开一位，损失与 `F.cross_entropy` 一致，并且均匀预测时损失等于 $\ln V$。

Implement sampling of pretraining data, the per-token loss with ignored positions and two small estimation functions. The tests verify that the target is exactly the input shifted by one, that the loss agrees with `F.cross_entropy`, and that a uniform prediction gives a loss of $\ln V$.

In [ ]:
import math
import torch
import torch.nn.functional as F


def make_batch(data, block_size, batch_size, generator=None):
    """Cut training examples out of one long token sequence.

    data: 1-D long tensor of token ids.
    Pick batch_size random start positions i with torch.randint(len(data) - block_size, (batch_size,), generator=generator).
    Returns (x, y), both (batch_size, block_size):
      x[b] = data[i : i + block_size]
      y[b] = data[i + 1 : i + block_size + 1]      the same text shifted by one token
    """
    # TODO 1
    raise NotImplementedError


def lm_loss(logits, targets, ignore_index=-100):
    """Next-token loss. logits: (B, T, V), targets: (B, T).

    Mean negative log-likelihood over all positions whose target is not ignore_index.
    Write it without F.cross_entropy: log-softmax, pick the target, mask, average.
    """
    # TODO 2
    raise NotImplementedError


def expected_initial_loss(vocab_size):
    """The loss of a model that predicts every token with equal probability."""
    # TODO 3
    raise NotImplementedError


def steps_per_epoch(n_tokens, block_size, batch_size):
    """How many optimizer steps it takes to see n_tokens tokens once (round down)."""
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
import torch.nn.functional as F

data = torch.arange(1000)
g = torch.Generator().manual_seed(0)
x, y = make_batch(data, block_size=16, batch_size=8, generator=g)
assert x.shape == (8, 16) and y.shape == (8, 16)
assert torch.equal(y, x + 1), "y is x shifted by one position"
assert torch.equal(x[:, 1:], y[:, :-1]), "the target at position t is the input at position t + 1"
assert len(set(x[:, 0].tolist())) > 1, "start positions should be random"

torch.manual_seed(0)
B, T, V = 4, 9, 50
logits = torch.randn(B, T, V, dtype=torch.float64)
targets = torch.randint(0, V, (B, T))
ref = F.cross_entropy(logits.view(-1, V), targets.view(-1))
assert torch.allclose(lm_loss(logits, targets), ref, atol=1e-10), "differs from F.cross_entropy"

masked = targets.clone()
masked[:, :4] = -100                                    # pretend the first four positions carry no target
ref_m = F.cross_entropy(logits.view(-1, V), masked.view(-1), ignore_index=-100)
assert torch.allclose(lm_loss(logits, masked), ref_m, atol=1e-10), "ignored positions must not count, neither in the sum nor in the average"

assert abs(expected_initial_loss(50257) - 10.8249) < 1e-3
uniform = torch.zeros(B, T, V, dtype=torch.float64)
assert abs(lm_loss(uniform, targets).item() - expected_initial_loss(V)) < 1e-10

assert steps_per_epoch(1_000_000, 1024, 32) == 30
assert steps_per_epoch(10_000_000_000, 1024, 512) == 19073
print("m41_batch: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

`x` 和 `y` 都用 `torch.stack` 把各个起点切出的片段叠起来，`y` 的切片比 `x` 整体右移一位。

Stack the slices from each start position with `torch.stack` for both `x` and `y`, where the slice of `y` is shifted right by one.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

用 `log_probs.gather(-1, targets.unsqueeze(-1))` 取出目标位置的对数概率。目标为 $-100$ 时下标不合法，先用 `clamp(min=0)` 换成任意合法的下标，再用掩码把这些位置清零。

Pick the log-probabilities of the targets with `log_probs.gather(-1, targets.unsqueeze(-1))`. An index of $-100$ is invalid, so first replace it with any valid index through `clamp(min=0)` and then zero these positions with the mask.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

取平均时分母是未被忽略的位置数，而不是全部位置数。

The average divides by the number of positions that are not ignored, not by all positions.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
import torch.nn.functional as F


def make_batch(data, block_size, batch_size, generator=None):
    ix = torch.randint(len(data) - block_size, (batch_size,), generator=generator)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y


def lm_loss(logits, targets, ignore_index=-100):
    log_probs = torch.log_softmax(logits, dim=-1)                       # (B, T, V)
    keep = targets != ignore_index                                      # (B, T)
    safe = targets.clamp(min=0)                                         # any valid index where the target is ignored
    picked = log_probs.gather(-1, safe.unsqueeze(-1)).squeeze(-1)       # (B, T)
    return -(picked * keep).sum() / keep.sum()


def expected_initial_loss(vocab_size):
    return math.log(vocab_size)


def steps_per_epoch(n_tokens, block_size, batch_size):
    return n_tokens // (block_size * batch_size)
```

</details>

## 2. 手写 AdamW 与学习率调度 / AdamW and the learning-rate schedule by hand

不使用 `torch.optim` 写出 AdamW 的一步更新，再实现带预热的余弦学习率调度。测试会连续 20 步与 `torch.optim.AdamW` 逐步对比，并验证第一步的更新量与梯度大小无关。

Write one AdamW update without `torch.optim` and implement the cosine learning-rate schedule with warm-up. The tests compare with `torch.optim.AdamW` step by step for 20 steps and verify that the first update does not depend on the size of the gradient.

In [ ]:
import math
import torch


def adamw_step(p, grad, m, v, t, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    """One AdamW update, without torch.optim.

    p, grad, m, v: tensors of the same shape (parameter, its gradient, first and second moment).
    t: the step number, starting at 1.
    Returns the new (p, m, v). Do not modify the inputs in place.
      p <- p - lr * weight_decay * p                     (decoupled weight decay)
      m <- b1 * m + (1 - b1) * grad
      v <- b2 * v + (1 - b2) * grad^2
      m_hat = m / (1 - b1^t),  v_hat = v / (1 - b2^t)    (bias correction)
      p <- p - lr * m_hat / (sqrt(v_hat) + eps)
    """
    # TODO 1
    raise NotImplementedError


def lr_schedule(step, max_lr, warmup_steps, total_steps, min_lr=0.0):
    """Linear warm-up followed by cosine decay.

    step 0 .. warmup_steps-1:  max_lr * (step + 1) / warmup_steps
    afterwards:                min_lr + 0.5 * (max_lr - min_lr) * (1 + cos(pi * progress))
                               with progress = (step - warmup_steps) / (total_steps - warmup_steps)
    at or beyond total_steps:  min_lr
    """
    # TODO 2
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch

torch.manual_seed(0)
w0 = torch.randn(5, 3, dtype=torch.float64)
target = torch.randn(5, 3, dtype=torch.float64)

# reference: torch.optim.AdamW on a simple quadratic loss
w_ref = w0.clone().requires_grad_(True)
opt = torch.optim.AdamW([w_ref], lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.1)
p, m, v = w0.clone(), torch.zeros_like(w0), torch.zeros_like(w0)
for t in range(1, 21):
    loss = ((w_ref - target) ** 2).sum()
    opt.zero_grad(); loss.backward(); opt.step()
    grad = 2 * (p - target)
    before = p.clone()
    p, m, v = adamw_step(p, grad, m, v, t, lr=1e-2, weight_decay=0.1)
    assert not torch.equal(before, p)
    assert torch.allclose(p, w_ref.detach(), atol=1e-10), f"step {t}: differs from torch.optim.AdamW"

# the very first step moves every parameter by about lr, whatever the size of its gradient
p1, _, _ = adamw_step(torch.zeros(3, dtype=torch.float64), torch.tensor([1e-4, 1.0, -1e4], dtype=torch.float64),
                      torch.zeros(3, dtype=torch.float64), torch.zeros(3, dtype=torch.float64), 1, lr=0.01, weight_decay=0.0)
assert torch.allclose(p1, torch.tensor([-0.01, -0.01, 0.01], dtype=torch.float64), atol=1e-5), "Adam normalizes the step size per parameter"

lr = lambda s: lr_schedule(s, max_lr=1.0, warmup_steps=10, total_steps=110, min_lr=0.1)
assert abs(lr(0) - 0.1) < 1e-12 and abs(lr(4) - 0.5) < 1e-12 and abs(lr(9) - 1.0) < 1e-12, "linear warm-up"
assert abs(lr(10) - 1.0) < 1e-12, "the decay starts at the maximum"
assert abs(lr(60) - 0.55) < 1e-12, "halfway through the decay the rate is halfway between max and min"
assert abs(lr(110) - 0.1) < 1e-12 and lr(500) == 0.1
assert all(lr(s) >= lr(s + 1) for s in range(10, 110)), "monotone decay after warm-up"
print("m41_adamw: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

按函数说明中的五行公式的顺序逐行写即可。权重衰减先做，并且用的是更新前的参数。

Write the five formulas of the docstring line by line in the given order. Weight decay comes first and uses the parameter before the update.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

不要原地修改输入的张量，用 `p = p - ...` 而不是 `p -= ...`。

Do not modify the input tensors in place. Write `p = p - ...` and not `p -= ...`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

学习率调度分三段，即预热、余弦下降和结束之后。先判断处在哪一段。

The schedule has three parts, warm-up, cosine decay and after the end. First decide which part the step is in.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def adamw_step(p, grad, m, v, t, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
    b1, b2 = betas
    p = p - lr * weight_decay * p                       # shrink the weights directly, not through the gradient
    m = b1 * m + (1 - b1) * grad                        # running mean of the gradient
    v = b2 * v + (1 - b2) * grad * grad                 # running mean of the squared gradient
    m_hat = m / (1 - b1 ** t)                           # both start at 0, so early values are too small
    v_hat = v / (1 - b2 ** t)
    p = p - lr * m_hat / (v_hat.sqrt() + eps)
    return p, m, v


def lr_schedule(step, max_lr, warmup_steps, total_steps, min_lr=0.0):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))
```

</details>

## 3. 预训练的规模估算与概念题 / Scale estimates and concept questions on pretraining

在纸上完成。
(a) 上下文长度 $T = 1024$，batch 大小为 512。每一步有多少个 token 参与训练？产生多少个训练信号？
(b) 语料共有 $3 \times 10^{11}$ 个 token。按 (a) 的设置，把语料过一遍需要多少步？
(c) 词表大小为 10 万的模型，训练刚开始时损失应当在什么数值附近？
(d) 证明 Adam 第一步的更新量等于 $-\eta \cdot \mathrm{sign}(g_1)$（忽略 $\epsilon$ 和权重衰减）。
(e) 用两三句话解释模型为什么有知识截止时间。

Work on paper.
(a) The context length is $T = 1024$ and the batch size 512. How many tokens take part in each step? How many training signals arise?
(b) The corpus has $3 \times 10^{11}$ tokens. With the setting of (a), how many steps does one pass over the corpus take?
(c) Around what value should the loss of a model with a vocabulary of 100 thousand start?
(d) Show that the first update of Adam equals $-\eta \cdot \mathrm{sign}(g_1)$ (ignoring $\epsilon$ and weight decay).
(e) Explain in two or three sentences why a model has a knowledge cutoff.

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 每个位置都预测它的下一个 token。

(a) Every position predicts its next token.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(d) 写出 $m_1$、$v_1$ 以及它们的偏差修正。

(d) Write out $m_1$, $v_1$ and their bias corrections.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 每步有 $1024 \times 512 = 524288$ 个 token，每个位置产生一个训练信号，所以也是 524288 个。

(b) $3 \times 10^{11} / 524288 \approx 5.7 \times 10^{5}$，约 57 万步。

(c) $\ln 10^{5} = 5 \ln 10 \approx 11.51$。

(d) $m_1 = (1 - \beta_1) g_1$，修正后 $\hat m_1 = g_1$。$v_1 = (1 - \beta_2) g_1^2$，修正后 $\hat v_1 = g_1^2$。所以更新量为 $-\eta\, g_1 / \sqrt{g_1^2} = -\eta\, g_1 / |g_1| = -\eta\, \mathrm{sign}(g_1)$。

(e) 模型的知识全部来自训练语料，语料的收集在某个时间点结束。训练完成后参数不再变化，所以此后发生的事情模型无从得知，除非在推理时把相关信息写进输入。

---

(a) Each step has $1024 \times 512 = 524288$ tokens, and every position gives one training signal, so 524288 as well.

(b) $3 \times 10^{11} / 524288 \approx 5.7 \times 10^{5}$, about 570 thousand steps.

(c) $\ln 10^{5} = 5 \ln 10 \approx 11.51$.

(d) $m_1 = (1 - \beta_1) g_1$, and after correction $\hat m_1 = g_1$. $v_1 = (1 - \beta_2) g_1^2$, and after correction $\hat v_1 = g_1^2$. The update is therefore $-\eta\, g_1 / \sqrt{g_1^2} = -\eta\, g_1 / |g_1| = -\eta\, \mathrm{sign}(g_1)$.

(e) All of a model's knowledge comes from its training corpus, whose collection ends at some point in time. After training the parameters no longer change, so the model cannot know what happened afterwards unless the information is written into the input at inference.

</details>